In [ ]:
from pathlib import Path
import os


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in (start, *start.parents):
        if (path / "interface_analyzer" / "reproducibility").exists():
            return path
    raise RuntimeError("Could not find repository root containing interface_analyzer/reproducibility")


PROJECT_ROOT = find_repo_root()
REPRO_DIR = PROJECT_ROOT / "interface_analyzer" / "reproducibility"
DATASET_DIR = REPRO_DIR / "dataset"
LOCAL_OUTPUT_DIR = REPRO_DIR / "_local_outputs"
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Full manuscript-scale post-processing files are intentionally not bundled.
# Set INTERFACE_ANALYZER_DATA to the directory containing those generated files.
FULL_DATA_ROOT = Path(os.environ.get("INTERFACE_ANALYZER_DATA", LOCAL_OUTPUT_DIR)).expanduser()
FULL_DATA_ROOT.mkdir(parents=True, exist_ok=True)

# For quick local CFG tests this defaults to the bundled sample dataset.
CFG_DIR = Path(os.environ.get("INTERFACE_ANALYZER_CFG_DIR", DATASET_DIR)).expanduser()

print("Project root:", PROJECT_ROOT)
print("Bundled CFG dataset:", DATASET_DIR)
print("Analysis data root:", FULL_DATA_ROOT)
print("CFG input dir:", CFG_DIR)


In [ ]:
import itertools
from pathlib import Path

import numpy as np
import pandas as pd

from interface_analyzer import analyze_cfm_fit_sensitivity


# =========================================================
# 1. User settings
# =========================================================
SAVE_DIR = FULL_DATA_ROOT
OUTPUT_DIR = Path("./k_sensitivity_repeat_stats")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# analyze_cfm_fit_sensitivity settings
K2_MIN = 5e-3
MIN_POINTS = 5
L_MIN_INTERFACE = 4
THROUGH_ORIGIN = True

# Whether to save intermediate per-orientation fit results for each file
SAVE_INDIVIDUAL_FIT_FILES = False


# =========================================================
# 2. Solve gamma0, delta1, delta2 from three stiffness values
#    Sequence: 100[010]; 110[001]; 110[1-10]
# =========================================================
def solve_gamma_delta(b1, b2, b3):
    """
    Solves for gamma_0, delta_1, and delta_2 given three orientation stiffness values.
    Sequence:
        b1 -> 100[010]
        b2 -> 110[001]
        b3 -> 110[1-10]
    """
    A = np.array([
        [1.0, -18/5,   -80/7],
        [1.0, -21/10,  365/14],
        [1.0,  39/10,  155/14]
    ], dtype=float)

    b = np.array([b1, b2, b3], dtype=float)
    sol = np.linalg.solve(A, b)

    g0 = sol[0]
    d1 = sol[1] / g0
    d2 = sol[2] / g0
    return g0, d1, d2


# =========================================================
# 3. File lists
#    You can freely adjust names here
# =========================================================
files_100_010 = [
    SAVE_DIR / "100_010" / "100_010_LOP_grid_2_5_d_6_0_10000_frames_k2.dat",
    SAVE_DIR / "100_010" / "100_010_LOP_grid_2_5_d_6_0_10000_frames_k2_repeat1.dat",
    SAVE_DIR / "100_010" / "100_010_LOP_grid_2_5_d_6_0_10000_frames_k2_repeat2.dat",
]

files_110_001 = [
    SAVE_DIR / "110_001" / "110_001_LOP_grid_2_5_d_6_0_10000_frames_k2.dat",
    SAVE_DIR / "110_001" / "110_001_LOP_grid_2_5_d_6_0_10000_frames_k2_repeat1.dat",
    SAVE_DIR / "110_001" / "110_001_LOP_grid_2_5_d_6_0_10000_frames_k2_repeat2.dat",
]

files_110_1m10 = [
    SAVE_DIR / "110_1-10" / "110_1-10_LOP_grid_2_5_d_6_0_10000_frames_k2.dat",
    SAVE_DIR / "110_1-10" / "110_1-10_LOP_grid_2_5_d_6_0_10000_frames_k2_repeat1.dat",
    SAVE_DIR / "110_1-10" / "110_1-10_LOP_grid_2_5_d_6_0_10000_frames_k2_repeat2.dat",
]


# =========================================================
# 4. Helper: run fit sensitivity for one k2 file
# =========================================================
def run_fit_sensitivity(k2_file):
    """
    Run analyze_cfm_fit_sensitivity on one file and return a DataFrame:
        N_Points, stiffness, R2
    """
    fit_results = analyze_cfm_fit_sensitivity(
        k2_file,
        k2_min=K2_MIN,
        min_points=MIN_POINTS,
        L_min_interface=L_MIN_INTERFACE,
        through_origin=THROUGH_ORIGIN
    )

    df = pd.DataFrame({
        "N_Points": fit_results["n_points"].astype(int),
        "stiffness": fit_results["stiffness"],
        "R2": fit_results["r2"]
    })

    return df


# =========================================================
# 5. Precompute fit sensitivity results for all individual files
#    This avoids re-reading / re-fitting the same file many times
# =========================================================
all_files = {
    "100_010": files_100_010,
    "110_001": files_110_001,
    "110_1-10": files_110_1m10,
}

fit_cache = {}

for orient, flist in all_files.items():
    for f in flist:
        if not f.exists():
            raise FileNotFoundError(f"File not found: {f}")

        print(f"Processing fit sensitivity: {f}")
        df_fit = run_fit_sensitivity(f)
        fit_cache[str(f)] = df_fit.copy()

        if SAVE_INDIVIDUAL_FIT_FILES:
            out_name = OUTPUT_DIR / f"{f.stem}_fit_sensitivity.dat"
            np.savetxt(
                out_name,
                np.c_[df_fit["N_Points"], df_fit["stiffness"], df_fit["R2"]],
                fmt="%d %.8e %.8f",
                header="N_Points_Used | Stiffness_Slope | R2_Coefficient",
                comments="# "
            )


# =========================================================
# 6. Enumerate all 27 combinations
# =========================================================
combo_results = []

combo_id = 0
for f1, f2, f3 in itertools.product(files_100_010, files_110_001, files_110_1m10):
    combo_id += 1

    df1 = fit_cache[str(f1)].rename(columns={"stiffness": "b1", "R2": "R2_b1"})
    df2 = fit_cache[str(f2)].rename(columns={"stiffness": "b2", "R2": "R2_b2"})
    df3 = fit_cache[str(f3)].rename(columns={"stiffness": "b3", "R2": "R2_b3"})

    # Merge on N_Points; only keep common N_Points across the three orientations
    df_merged = df1.merge(df2, on="N_Points").merge(df3, on="N_Points")

    if df_merged.empty:
        print(f"Warning: empty merged result for combo {combo_id}")
        continue

    for _, row in df_merged.iterrows():
        try:
            g0, d1, d2 = solve_gamma_delta(row["b1"], row["b2"], row["b3"])
        except np.linalg.LinAlgError:
            print(f"Warning: singular solve in combo {combo_id}, N_Points={row['N_Points']}")
            continue

        combo_results.append({
            "combo_id": combo_id,
            "file_100_010": f1.name,
            "file_110_001": f2.name,
            "file_110_1-10": f3.name,
            "N_Points": int(row["N_Points"]),
            "b1": row["b1"],
            "b2": row["b2"],
            "b3": row["b3"],
            "R2_b1": row["R2_b1"],
            "R2_b2": row["R2_b2"],
            "R2_b3": row["R2_b3"],
            "R2_avg": (row["R2_b1"] + row["R2_b2"] + row["R2_b3"]) / 3.0,
            "gamma_0": g0,
            "delta_1": d1,
            "delta_2": d2
        })

df_all_combos = pd.DataFrame(combo_results)

if df_all_combos.empty:
    raise RuntimeError("No valid combination results were generated.")


# =========================================================
# 7. Statistics over the 27 combinations at each N_Points
# =========================================================
df_stats = (
    df_all_combos
    .groupby("N_Points", as_index=False)
    .agg(
        gamma_0_mean=("gamma_0", "mean"),
        gamma_0_std=("gamma_0", "std"),
        delta_1_mean=("delta_1", "mean"),
        delta_1_std=("delta_1", "std"),
        delta_2_mean=("delta_2", "mean"),
        delta_2_std=("delta_2", "std"),
        R2_avg_mean=("R2_avg", "mean"),
        R2_avg_std=("R2_avg", "std"),
        n_combinations=("combo_id", "count")
    )
    .sort_values("N_Points")
)

# 如果某个 N_Points 只有 1 个组合，则 std 会是 NaN，这里改成 0
std_cols = [
    "gamma_0_std", "delta_1_std", "delta_2_std", "R2_avg_std"
]
df_stats[std_cols] = df_stats[std_cols].fillna(0.0)


# =========================================================
# 8. Save outputs
# =========================================================
all_combo_csv = OUTPUT_DIR / "all_27_combinations_gamma_delta_results.csv"
stats_csv = OUTPUT_DIR / "gamma_delta_statistics_over_27_combinations.csv"
stats_dat = OUTPUT_DIR / "gamma_delta_statistics_over_27_combinations.dat"

df_all_combos.to_csv(all_combo_csv, index=False)
df_stats.to_csv(stats_csv, index=False)

np.savetxt(
    stats_dat,
    df_stats[[
        "N_Points",
        "gamma_0_mean", "gamma_0_std",
        "delta_1_mean", "delta_1_std",
        "delta_2_mean", "delta_2_std",
        "R2_avg_mean", "R2_avg_std",
        "n_combinations"
    ]].values,
    fmt=["%d", "%.8e", "%.8e", "%.8e", "%.8e", "%.8e", "%.8e", "%.8f", "%.8f", "%d"],
    header=(
        "N_Points "
        "gamma_0_mean gamma_0_std "
        "delta_1_mean delta_1_std "
        "delta_2_mean delta_2_std "
        "R2_avg_mean R2_avg_std "
        "n_combinations"
    ),
    comments="# "
)


# =========================================================
# 9. Print summary
# =========================================================
print("\nAll 27-combination results saved to:")
print(all_combo_csv)

print("\nStatistics over combinations saved to:")
print(stats_csv)
print(stats_dat)

print("\nAggregated statistics:")
print(df_stats.to_string(index=False, formatters={
    "gamma_0_mean": "{:,.6e}".format,
    "gamma_0_std": "{:,.6e}".format,
    "delta_1_mean": "{:.6f}".format,
    "delta_1_std": "{:.6f}".format,
    "delta_2_mean": "{:.6f}".format,
    "delta_2_std": "{:.6f}".format,
    "R2_avg_mean": "{:.6f}".format,
    "R2_avg_std": "{:.6f}".format,
}))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path

# =========================================================
# If running as a separate script, uncomment and set paths:
# =========================================================
# OUTPUT_DIR = Path("./k_sensitivity_repeat_stats")
# df_all_combos = pd.read_csv(OUTPUT_DIR / "all_27_combinations_gamma_delta_results.csv")
# df_stats = pd.read_csv(OUTPUT_DIR / "gamma_delta_statistics_over_27_combinations.csv")

# Ensure output dir exists
OUTPUT_DIR = Path("./k_sensitivity_repeat_stats")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# =========================================================
# Plot settings
# =========================================================
FIGSIZE = (15, 4.8)
RAW_ALPHA_LINE = 0.22
RAW_ALPHA_SCATTER = 0.28
RAW_LINEWIDTH = 1.0
RAW_MARKERSIZE = 16

MEAN_LINEWIDTH = 2.5
MEAN_MARKERSIZE = 5
ERRORBAR_CAPSIZE = 3

SAVE_PNG = OUTPUT_DIR / "gamma_delta_triptych_with_raw_and_errorbars.png"
SAVE_PDF = OUTPUT_DIR / "gamma_delta_triptych_with_raw_and_errorbars.pdf"


# =========================================================
# Optional: sort data to ensure lines are drawn correctly
# =========================================================
df_all_combos = df_all_combos.sort_values(["combo_id", "N_Points"]).copy()
df_stats = df_stats.sort_values("N_Points").copy()


# =========================================================
# Create figure
# =========================================================
fig, axes = plt.subplots(1, 3, figsize=FIGSIZE, sharex=True)
ax1, ax2, ax3 = axes

# -----------------------------
# Panel 1: gamma_0
# -----------------------------
for combo_id, grp in df_all_combos.groupby("combo_id"):
    grp = grp.sort_values("N_Points")
    ax1.plot(
        grp["N_Points"], grp["gamma_0"],
        "-", linewidth=RAW_LINEWIDTH, alpha=RAW_ALPHA_LINE
    )
    ax1.scatter(
        grp["N_Points"], grp["gamma_0"],
        s=RAW_MARKERSIZE, alpha=RAW_ALPHA_SCATTER
    )

ax1.errorbar(
    df_stats["N_Points"], df_stats["gamma_0_mean"],
    yerr=df_stats["gamma_0_std"],
    fmt="-o", linewidth=MEAN_LINEWIDTH, markersize=MEAN_MARKERSIZE,
    capsize=ERRORBAR_CAPSIZE
)
ax1.set_xlabel("Number of k points used")
ax1.set_ylabel(r"$\gamma_0$")
ax1.set_title(r"$\gamma_0$")
ax1.grid(True, alpha=0.3)


# -----------------------------
# Panel 2: delta_1
# -----------------------------
for combo_id, grp in df_all_combos.groupby("combo_id"):
    grp = grp.sort_values("N_Points")
    ax2.plot(
        grp["N_Points"], grp["delta_1"],
        "-", linewidth=RAW_LINEWIDTH, alpha=RAW_ALPHA_LINE
    )
    ax2.scatter(
        grp["N_Points"], grp["delta_1"],
        s=RAW_MARKERSIZE, alpha=RAW_ALPHA_SCATTER
    )

ax2.errorbar(
    df_stats["N_Points"], df_stats["delta_1_mean"],
    yerr=df_stats["delta_1_std"],
    fmt="-o", linewidth=MEAN_LINEWIDTH, markersize=MEAN_MARKERSIZE,
    capsize=ERRORBAR_CAPSIZE
)
ax2.set_xlabel("Number of k points used")
ax2.set_ylabel(r"$\delta_1$")
ax2.set_title(r"$\delta_1$")
ax2.grid(True, alpha=0.3)


# -----------------------------
# Panel 3: delta_2
# -----------------------------
for combo_id, grp in df_all_combos.groupby("combo_id"):
    grp = grp.sort_values("N_Points")
    ax3.plot(
        grp["N_Points"], grp["delta_2"],
        "-", linewidth=RAW_LINEWIDTH, alpha=RAW_ALPHA_LINE
    )
    ax3.scatter(
        grp["N_Points"], grp["delta_2"],
        s=RAW_MARKERSIZE, alpha=RAW_ALPHA_SCATTER
    )

ax3.errorbar(
    df_stats["N_Points"], df_stats["delta_2_mean"],
    yerr=df_stats["delta_2_std"],
    fmt="-o", linewidth=MEAN_LINEWIDTH, markersize=MEAN_MARKERSIZE,
    capsize=ERRORBAR_CAPSIZE
)
ax3.set_xlabel("Number of k points used")
ax3.set_ylabel(r"$\delta_2$")
ax3.set_title(r"$\delta_2$")
ax3.grid(True, alpha=0.3)


# =========================================================
# Final layout and save
# =========================================================
plt.tight_layout()
plt.savefig(SAVE_PNG, dpi=300, bbox_inches="tight")
plt.savefig(SAVE_PDF, bbox_inches="tight")
plt.show()

print(f"Saved figure to:\n  {SAVE_PNG}\n  {SAVE_PDF}")

In [ ]:
import itertools
from pathlib import Path

import numpy as np
import pandas as pd

from interface_analyzer import analyze_cfm_fit_sensitivity


# =========================================================
# 1. User settings
# =========================================================
SAVE_DIR = FULL_DATA_ROOT
OUTPUT_DIR = Path("./k_sensitivity_repeat_stats")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# analyze_cfm_fit_sensitivity settings
K2_MIN = 5e-3
MIN_POINTS = 5
L_MIN_INTERFACE = 4
THROUGH_ORIGIN = True

# Whether to save intermediate per-orientation fit results for each file
SAVE_INDIVIDUAL_FIT_FILES = False


# =========================================================
# 2. Solve gamma0, delta1, delta2 from three stiffness values
#    Sequence: 100[010]; 110[001]; 110[1-10]
# =========================================================
def solve_gamma_delta(b1, b2, b3):
    """
    Solves for gamma_0, delta_1, and delta_2 given three orientation stiffness values.
    Sequence:
        b1 -> 100[010]
        b2 -> 110[001]
        b3 -> 110[1-10]
    """
    A = np.array([
        [1.0, -18/5,   -80/7],
        [1.0, -21/10,  365/14],
        [1.0,  39/10,  155/14]
    ], dtype=float)

    b = np.array([b1, b2, b3], dtype=float)
    sol = np.linalg.solve(A, b)

    g0 = sol[0]
    d1 = sol[1] / g0
    d2 = sol[2] / g0
    return g0, d1, d2


# =========================================================
# 3. File lists
#    You can freely adjust names here
# =========================================================
files_100_010 = [
    SAVE_DIR / "100_010" / "100_010_LOP_grid_2_5_d_6_0_5000_frames_k2.dat",
    SAVE_DIR / "100_010" / "100_010_LOP_grid_2_5_d_6_0_5000_frames_k2_repeat1.dat",
    SAVE_DIR / "100_010" / "100_010_LOP_grid_2_5_d_6_0_5000_frames_k2_repeat2.dat",
]

files_110_001 = [
    SAVE_DIR / "110_001" / "110_001_LOP_grid_2_5_d_6_0_5000_frames_k2.dat",
    SAVE_DIR / "110_001" / "110_001_LOP_grid_2_5_d_6_0_5000_frames_k2_repeat1.dat",
    SAVE_DIR / "110_001" / "110_001_LOP_grid_2_5_d_6_0_5000_frames_k2_repeat2.dat",
]

files_110_1m10 = [
    SAVE_DIR / "110_1-10" / "110_1-10_LOP_grid_2_5_d_6_0_5000_frames_k2.dat",
    SAVE_DIR / "110_1-10" / "110_1-10_LOP_grid_2_5_d_6_0_5000_frames_k2_repeat1.dat",
    SAVE_DIR / "110_1-10" / "110_1-10_LOP_grid_2_5_d_6_0_5000_frames_k2_repeat2.dat",
]


# =========================================================
# 4. Helper: run fit sensitivity for one k2 file
# =========================================================
def run_fit_sensitivity(k2_file):
    """
    Run analyze_cfm_fit_sensitivity on one file and return a DataFrame:
        N_Points, stiffness, R2
    """
    fit_results = analyze_cfm_fit_sensitivity(
        k2_file,
        k2_min=K2_MIN,
        min_points=MIN_POINTS,
        L_min_interface=L_MIN_INTERFACE,
        through_origin=THROUGH_ORIGIN
    )

    df = pd.DataFrame({
        "N_Points": fit_results["n_points"].astype(int),
        "stiffness": fit_results["stiffness"],
        "R2": fit_results["r2"]
    })

    return df


# =========================================================
# 5. Precompute fit sensitivity results for all individual files
#    This avoids re-reading / re-fitting the same file many times
# =========================================================
all_files = {
    "100_010": files_100_010,
    "110_001": files_110_001,
    "110_1-10": files_110_1m10,
}

fit_cache = {}

for orient, flist in all_files.items():
    for f in flist:
        if not f.exists():
            raise FileNotFoundError(f"File not found: {f}")

        print(f"Processing fit sensitivity: {f}")
        df_fit = run_fit_sensitivity(f)
        fit_cache[str(f)] = df_fit.copy()

        if SAVE_INDIVIDUAL_FIT_FILES:
            out_name = OUTPUT_DIR / f"{f.stem}_fit_sensitivity.dat"
            np.savetxt(
                out_name,
                np.c_[df_fit["N_Points"], df_fit["stiffness"], df_fit["R2"]],
                fmt="%d %.8e %.8f",
                header="N_Points_Used | Stiffness_Slope | R2_Coefficient",
                comments="# "
            )


# =========================================================
# 6. Enumerate all 27 combinations
# =========================================================
combo_results = []

combo_id = 0
for f1, f2, f3 in itertools.product(files_100_010, files_110_001, files_110_1m10):
    combo_id += 1

    df1 = fit_cache[str(f1)].rename(columns={"stiffness": "b1", "R2": "R2_b1"})
    df2 = fit_cache[str(f2)].rename(columns={"stiffness": "b2", "R2": "R2_b2"})
    df3 = fit_cache[str(f3)].rename(columns={"stiffness": "b3", "R2": "R2_b3"})

    # Merge on N_Points; only keep common N_Points across the three orientations
    df_merged = df1.merge(df2, on="N_Points").merge(df3, on="N_Points")

    if df_merged.empty:
        print(f"Warning: empty merged result for combo {combo_id}")
        continue

    for _, row in df_merged.iterrows():
        try:
            g0, d1, d2 = solve_gamma_delta(row["b1"], row["b2"], row["b3"])
        except np.linalg.LinAlgError:
            print(f"Warning: singular solve in combo {combo_id}, N_Points={row['N_Points']}")
            continue

        combo_results.append({
            "combo_id": combo_id,
            "file_100_010": f1.name,
            "file_110_001": f2.name,
            "file_110_1-10": f3.name,
            "N_Points": int(row["N_Points"]),
            "b1": row["b1"],
            "b2": row["b2"],
            "b3": row["b3"],
            "R2_b1": row["R2_b1"],
            "R2_b2": row["R2_b2"],
            "R2_b3": row["R2_b3"],
            "R2_avg": (row["R2_b1"] + row["R2_b2"] + row["R2_b3"]) / 3.0,
            "gamma_0": g0,
            "delta_1": d1,
            "delta_2": d2
        })

df_all_combos = pd.DataFrame(combo_results)

if df_all_combos.empty:
    raise RuntimeError("No valid combination results were generated.")


# =========================================================
# 7. Statistics over the 27 combinations at each N_Points
# =========================================================
df_stats = (
    df_all_combos
    .groupby("N_Points", as_index=False)
    .agg(
        gamma_0_mean=("gamma_0", "mean"),
        gamma_0_std=("gamma_0", "std"),
        delta_1_mean=("delta_1", "mean"),
        delta_1_std=("delta_1", "std"),
        delta_2_mean=("delta_2", "mean"),
        delta_2_std=("delta_2", "std"),
        R2_avg_mean=("R2_avg", "mean"),
        R2_avg_std=("R2_avg", "std"),
        n_combinations=("combo_id", "count")
    )
    .sort_values("N_Points")
)

# 如果某个 N_Points 只有 1 个组合，则 std 会是 NaN，这里改成 0
std_cols = [
    "gamma_0_std", "delta_1_std", "delta_2_std", "R2_avg_std"
]
df_stats[std_cols] = df_stats[std_cols].fillna(0.0)


# =========================================================
# 8. Save outputs
# =========================================================
all_combo_csv = OUTPUT_DIR / "all_27_combinations_gamma_delta_results.csv"
stats_csv = OUTPUT_DIR / "gamma_delta_statistics_over_27_combinations.csv"
stats_dat = OUTPUT_DIR / "gamma_delta_statistics_over_27_combinations.dat"

df_all_combos.to_csv(all_combo_csv, index=False)
df_stats.to_csv(stats_csv, index=False)

np.savetxt(
    stats_dat,
    df_stats[[
        "N_Points",
        "gamma_0_mean", "gamma_0_std",
        "delta_1_mean", "delta_1_std",
        "delta_2_mean", "delta_2_std",
        "R2_avg_mean", "R2_avg_std",
        "n_combinations"
    ]].values,
    fmt=["%d", "%.8e", "%.8e", "%.8e", "%.8e", "%.8e", "%.8e", "%.8f", "%.8f", "%d"],
    header=(
        "N_Points "
        "gamma_0_mean gamma_0_std "
        "delta_1_mean delta_1_std "
        "delta_2_mean delta_2_std "
        "R2_avg_mean R2_avg_std "
        "n_combinations"
    ),
    comments="# "
)


# =========================================================
# 9. Print summary
# =========================================================
print("\nAll 27-combination results saved to:")
print(all_combo_csv)

print("\nStatistics over combinations saved to:")
print(stats_csv)
print(stats_dat)

print("\nAggregated statistics:")
print(df_stats.to_string(index=False, formatters={
    "gamma_0_mean": "{:,.6e}".format,
    "gamma_0_std": "{:,.6e}".format,
    "delta_1_mean": "{:.6f}".format,
    "delta_1_std": "{:.6f}".format,
    "delta_2_mean": "{:.6f}".format,
    "delta_2_std": "{:.6f}".format,
    "R2_avg_mean": "{:.6f}".format,
    "R2_avg_std": "{:.6f}".format,
}))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path

# =========================================================
# If running as a separate script, uncomment and set paths:
# =========================================================
# OUTPUT_DIR = Path("./k_sensitivity_repeat_stats")
# df_all_combos = pd.read_csv(OUTPUT_DIR / "all_27_combinations_gamma_delta_results.csv")
# df_stats = pd.read_csv(OUTPUT_DIR / "gamma_delta_statistics_over_27_combinations.csv")

# Ensure output dir exists
OUTPUT_DIR = Path("./k_sensitivity_repeat_stats_5000frams")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# =========================================================
# Plot settings
# =========================================================
FIGSIZE = (15, 4.8)
RAW_ALPHA_LINE = 0.22
RAW_ALPHA_SCATTER = 0.28
RAW_LINEWIDTH = 1.0
RAW_MARKERSIZE = 16

MEAN_LINEWIDTH = 2.5
MEAN_MARKERSIZE = 5
ERRORBAR_CAPSIZE = 3

SAVE_PNG = OUTPUT_DIR / "gamma_delta_triptych_with_raw_and_errorbars.png"
SAVE_PDF = OUTPUT_DIR / "gamma_delta_triptych_with_raw_and_errorbars.pdf"


# =========================================================
# Optional: sort data to ensure lines are drawn correctly
# =========================================================
df_all_combos = df_all_combos.sort_values(["combo_id", "N_Points"]).copy()
df_stats = df_stats.sort_values("N_Points").copy()


# =========================================================
# Create figure
# =========================================================
fig, axes = plt.subplots(1, 3, figsize=FIGSIZE, sharex=True)
ax1, ax2, ax3 = axes

# -----------------------------
# Panel 1: gamma_0
# -----------------------------
for combo_id, grp in df_all_combos.groupby("combo_id"):
    grp = grp.sort_values("N_Points")
    ax1.plot(
        grp["N_Points"], grp["gamma_0"],
        "-", linewidth=RAW_LINEWIDTH, alpha=RAW_ALPHA_LINE
    )
    ax1.scatter(
        grp["N_Points"], grp["gamma_0"],
        s=RAW_MARKERSIZE, alpha=RAW_ALPHA_SCATTER
    )

ax1.errorbar(
    df_stats["N_Points"], df_stats["gamma_0_mean"],
    yerr=df_stats["gamma_0_std"],
    fmt="-o", linewidth=MEAN_LINEWIDTH, markersize=MEAN_MARKERSIZE,
    capsize=ERRORBAR_CAPSIZE
)
ax1.set_xlabel("Number of k points used")
ax1.set_ylabel(r"$\gamma_0$")
ax1.set_title(r"$\gamma_0$")
ax1.grid(True, alpha=0.3)


# -----------------------------
# Panel 2: delta_1
# -----------------------------
for combo_id, grp in df_all_combos.groupby("combo_id"):
    grp = grp.sort_values("N_Points")
    ax2.plot(
        grp["N_Points"], grp["delta_1"],
        "-", linewidth=RAW_LINEWIDTH, alpha=RAW_ALPHA_LINE
    )
    ax2.scatter(
        grp["N_Points"], grp["delta_1"],
        s=RAW_MARKERSIZE, alpha=RAW_ALPHA_SCATTER
    )

ax2.errorbar(
    df_stats["N_Points"], df_stats["delta_1_mean"],
    yerr=df_stats["delta_1_std"],
    fmt="-o", linewidth=MEAN_LINEWIDTH, markersize=MEAN_MARKERSIZE,
    capsize=ERRORBAR_CAPSIZE
)
ax2.set_xlabel("Number of k points used")
ax2.set_ylabel(r"$\delta_1$")
ax2.set_title(r"$\delta_1$")
ax2.grid(True, alpha=0.3)


# -----------------------------
# Panel 3: delta_2
# -----------------------------
for combo_id, grp in df_all_combos.groupby("combo_id"):
    grp = grp.sort_values("N_Points")
    ax3.plot(
        grp["N_Points"], grp["delta_2"],
        "-", linewidth=RAW_LINEWIDTH, alpha=RAW_ALPHA_LINE
    )
    ax3.scatter(
        grp["N_Points"], grp["delta_2"],
        s=RAW_MARKERSIZE, alpha=RAW_ALPHA_SCATTER
    )

ax3.errorbar(
    df_stats["N_Points"], df_stats["delta_2_mean"],
    yerr=df_stats["delta_2_std"],
    fmt="-o", linewidth=MEAN_LINEWIDTH, markersize=MEAN_MARKERSIZE,
    capsize=ERRORBAR_CAPSIZE
)
ax3.set_xlabel("Number of k points used")
ax3.set_ylabel(r"$\delta_2$")
ax3.set_title(r"$\delta_2$")
ax3.grid(True, alpha=0.3)


# =========================================================
# Final layout and save
# =========================================================
plt.tight_layout()
plt.savefig(SAVE_PNG, dpi=300, bbox_inches="tight")
plt.savefig(SAVE_PDF, bbox_inches="tight")
plt.show()

print(f"Saved figure to:\n  {SAVE_PNG}\n  {SAVE_PDF}")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from matplotlib.ticker import MaxNLocator

# =========================================================
# Global plotting style
# =========================================================
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 15,

    "axes.labelsize": 16,     # 坐标轴label更大
    "axes.titlesize": 17,

    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 13,
    "axes.linewidth": 1.2,
    "xtick.direction": "in",
    "ytick.direction": "in",
})

# =========================================================
# Paths
# =========================================================
OUTPUT_DIR = Path("./k_sensitivity_repeat_stats")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ALL_COMBO_CSV = OUTPUT_DIR / "all_27_combinations_gamma_delta_results.csv"
STATS_CSV = OUTPUT_DIR / "gamma_delta_statistics_over_27_combinations.csv"

SAVE_PNG = OUTPUT_DIR / "gamma_epsilon_triptych_with_raw_and_errorbars_independent.png"
SAVE_PDF = OUTPUT_DIR / "gamma_epsilon_triptych_with_raw_and_errorbars_independent.pdf"

# =========================================================
# Read data
# =========================================================
df_all_combos = pd.read_csv(ALL_COMBO_CSV)
df_stats = pd.read_csv(STATS_CSV)

# Sort data to ensure lines are drawn correctly
df_all_combos = df_all_combos.sort_values(["combo_id", "N_Points"]).copy()
df_stats = df_stats.sort_values("N_Points").copy()

# =========================================================
# Unit conversion for gamma_0
# ---------------------------------------------------------
# Original gamma_0 is assumed in J/m^2-equivalent scale where
# multiplying by 100 gives mJ/m^2 as requested by user.
# =========================================================
gamma_scale = 1E20

df_all_combos["gamma_0_plot"] = df_all_combos["gamma_0"] * gamma_scale
df_stats["gamma_0_mean_plot"] = df_stats["gamma_0_mean"] * gamma_scale
df_stats["gamma_0_std_plot"] = df_stats["gamma_0_std"] * gamma_scale

# =========================================================
# Plot settings
# =========================================================
FIGSIZE = (15.5, 5.0)

RAW_ALPHA_LINE = 0.20
RAW_ALPHA_SCATTER = 0.25
RAW_LINEWIDTH = 1.0
RAW_MARKERSIZE = 14

MEAN_LINEWIDTH = 2.6
MEAN_MARKERSIZE = 5.5
ERRORBAR_CAPSIZE = 3

MEAN_COLOR = "black"
ERR_COLOR = "black"
ERR_ALPHA = 0.6

# =========================================================
# Create figure
# =========================================================
fig, axes = plt.subplots(1, 3, figsize=FIGSIZE, sharex=True)
ax1, ax2, ax3 = axes

# =========================================================
# Panel 1: gamma_0
# =========================================================
for combo_id, grp in df_all_combos.groupby("combo_id"):
    grp = grp.sort_values("N_Points")
    ax1.plot(
        grp["N_Points"], grp["gamma_0_plot"],
        "-", linewidth=RAW_LINEWIDTH, alpha=RAW_ALPHA_LINE
    )
    ax1.scatter(
        grp["N_Points"], grp["gamma_0_plot"],
        s=RAW_MARKERSIZE, alpha=RAW_ALPHA_SCATTER
    )

# draw error bars first so black mean line stays on top
ax1.errorbar(
    df_stats["N_Points"], df_stats["gamma_0_mean_plot"],
    yerr=df_stats["gamma_0_std_plot"],
    fmt="none",
    ecolor=ERR_COLOR,
    elinewidth=1.6,
    alpha=ERR_ALPHA,
    capsize=ERRORBAR_CAPSIZE,
    capthick=1.4,
    zorder=3
)

ax1.plot(
    df_stats["N_Points"], df_stats["gamma_0_mean_plot"],
    "-o",
    color=MEAN_COLOR,
    linewidth=MEAN_LINEWIDTH,
    markersize=MEAN_MARKERSIZE,
    zorder=4
)

ax1.set_xlabel("Number of k points used")
ax1.set_ylabel(r"$\gamma_0$ (mJ m$^{-2}$)")
ax1.set_title(r"$\gamma_0$")
ax1.grid(True, alpha=0.3)

# sparse y ticks
ax1.yaxis.set_major_locator(MaxNLocator(nbins=5))

# =========================================================
# Panel 2: epsilon_1
# =========================================================
for combo_id, grp in df_all_combos.groupby("combo_id"):
    grp = grp.sort_values("N_Points")
    ax2.plot(
        grp["N_Points"], grp["delta_1"],
        "-", linewidth=RAW_LINEWIDTH, alpha=RAW_ALPHA_LINE
    )
    ax2.scatter(
        grp["N_Points"], grp["delta_1"],
        s=RAW_MARKERSIZE, alpha=RAW_ALPHA_SCATTER
    )

ax2.errorbar(
    df_stats["N_Points"], df_stats["delta_1_mean"],
    yerr=df_stats["delta_1_std"],
    fmt="none",
    ecolor=ERR_COLOR,
    elinewidth=1.6,
    alpha=ERR_ALPHA,
    capsize=ERRORBAR_CAPSIZE,
    capthick=1.4,
    zorder=3
)

ax2.plot(
    df_stats["N_Points"], df_stats["delta_1_mean"],
    "-o",
    color=MEAN_COLOR,
    linewidth=MEAN_LINEWIDTH,
    markersize=MEAN_MARKERSIZE,
    zorder=4
)

ax2.set_xlabel("Number of k points used")
ax2.set_ylabel(r"$\varepsilon_1$")
ax2.set_title(r"$\varepsilon_1$")
ax2.grid(True, alpha=0.3)
ax2.yaxis.set_major_locator(MaxNLocator(nbins=5))

# =========================================================
# Panel 3: epsilon_2
# =========================================================
for combo_id, grp in df_all_combos.groupby("combo_id"):
    grp = grp.sort_values("N_Points")
    ax3.plot(
        grp["N_Points"], grp["delta_2"],
        "-", linewidth=RAW_LINEWIDTH, alpha=RAW_ALPHA_LINE
    )
    ax3.scatter(
        grp["N_Points"], grp["delta_2"],
        s=RAW_MARKERSIZE, alpha=RAW_ALPHA_SCATTER
    )

ax3.errorbar(
    df_stats["N_Points"], df_stats["delta_2_mean"],
    yerr=df_stats["delta_2_std"],
    fmt="none",
    ecolor=ERR_COLOR,
    elinewidth=1.6,
    alpha=ERR_ALPHA,
    capsize=ERRORBAR_CAPSIZE,
    capthick=1.4,
    zorder=3
)

ax3.plot(
    df_stats["N_Points"], df_stats["delta_2_mean"],
    "-o",
    color=MEAN_COLOR,
    linewidth=MEAN_LINEWIDTH,
    markersize=MEAN_MARKERSIZE,
    zorder=4
)

ax3.set_xlabel("Number of k points used")
ax3.set_ylabel(r"$\varepsilon_2$")
ax3.set_title(r"$\varepsilon_2$")
ax3.grid(True, alpha=0.3)
ax3.yaxis.set_major_locator(MaxNLocator(nbins=5))

# =========================================================
# Final layout and save
# =========================================================
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "gamma_epsilon_triptych_with_raw_and_errorbars_independent.svg", bbox_inches="tight")
plt.show()